# DBRepo Semantic Unit Mapping

This notebook applies the **SI Digital Framework** and **OMG Quantities** semantic concepts to the numerical columns in your DBRepo schema via the REST API. It uses a cleaned, modularized code structure.

In [1]:
import requests
from requests.auth import HTTPBasicAuth
from getpass import getpass

# --- Configuration ---
BASE_URL = "https://test.dbrepo.tuwien.ac.at/api/v1"
DATABASE_ID = "b23492bd-f66d-4663-a1f5-f296767dbbdc"

# (table_name, column_name) -> (concept_uri, unit_uri)
SEMANTIC_MAPPINGS = {
    ("accident", "easting"): (
        "http://www.wikidata.org/entity/Q192203",
        "https://si-digital-framework.org/SI/units/metre"
    ),
    ("accident", "northing"): (
        "http://www.wikidata.org/entity/Q192203",
        "https://si-digital-framework.org/SI/units/metre"
    ),
    ("accident", "number_of_vehicles"): (
        "http://www.wikidata.org/entity/Q28206497",
        "https://www.omg.org/spec/Commons/QuantitiesAndUnits/Count"
    ),
    ("person", "age"): (
        "http://www.wikidata.org/entity/Q185836",
        "https://www.omg.org/spec/Commons/QuantitiesAndUnits/Year"
    )
}

In [2]:
def get_database_info(session: requests.Session) -> dict:
    """Fetches DBRepo database metadata."""
    response = session.get(f"{BASE_URL}/database/{DATABASE_ID}")
    response.raise_for_status()
    return response.json()

def find_table_and_column(db_info: dict, table_name: str, col_name: str) -> tuple:
    """Finds table and column metadata IDs by their names."""
    target_table = next((t for t in db_info.get("tables", []) if t.get("name") == table_name), None)
    if not target_table:
        raise ValueError(f"Table '{table_name}' not found.")

    target_col = next((c for c in target_table.get("columns", []) if c.get("name") == col_name), None)
    if not target_col:
        raise ValueError(f"Column '{col_name}' not found in table '{table_name}'.")

    return target_table, target_col

def update_column_semantics(session: requests.Session, db_info: dict, table_name: str, 
                            col_name: str, concept_uri: str, unit_uri: str) -> bool:
    """Updates only the semantic metadata of a DBRepo column."""
    target_table, target_col = find_table_and_column(db_info, table_name, col_name)
    
    put_url = f"{BASE_URL}/database/{DATABASE_ID}/table/{target_table['id']}/column/{target_col['id']}"
    payload = {"concept_uri": concept_uri, "unit_uri": unit_uri}

    response = session.put(put_url, json=payload)
    
    if response.status_code in (200, 201, 202, 204):
        print(f"[✓] Successfully mapped {table_name}.{col_name} -> {unit_uri}")
        return True

    print(f"[X] Failed to update {table_name}.{col_name}: {response.status_code} - {response.text}")
    return False

In [3]:
# ==========================================
# Execution Block
# ==========================================

print("--- DBRepo Semantic Unit Mapping ---")
username = input("DBRepo username: ").strip()
password = getpass("DBRepo password: ")

# Setup Session
session = requests.Session()
session.auth = HTTPBasicAuth(username, password)
session.headers.update({"Accept": "application/json", "Content-Type": "application/json"})

try:
    db_info = get_database_info(session)
    success_count = 0
    print("\nApplying semantic unit mappings...")
    for (table, col), (concept, unit) in SEMANTIC_MAPPINGS.items():
        if update_column_semantics(session, db_info, table, col, concept, unit):
            success_count += 1

    print(f"\nMapping complete: {success_count}/{len(SEMANTIC_MAPPINGS)} columns updated.")
    
except requests.exceptions.RequestException as e:
    print(f"Error connecting to DBRepo: {e}")

--- DBRepo Semantic Unit Mapping ---

Applying semantic unit mappings...
[✓] Successfully mapped accident.easting -> https://si-digital-framework.org/SI/units/metre
[✓] Successfully mapped accident.northing -> https://si-digital-framework.org/SI/units/metre
[✓] Successfully mapped accident.number_of_vehicles -> https://www.omg.org/spec/Commons/QuantitiesAndUnits/Count
[✓] Successfully mapped person.age -> https://www.omg.org/spec/Commons/QuantitiesAndUnits/Year

Mapping complete: 4/4 columns updated.
